# GIADA Task 17d — da dove nasce la sensibilità numerica del teacher
Esperimento diagnostico nativo su stati, eventi e timing. Protocollo: `experiments/task17d_native_solver_mechanism_preregistration.md`. Gate C resta chiuso.

In [ ]:
from pathlib import Path
import base64,json,os,shutil,subprocess,sys,tempfile,uuid
from IPython.display import Javascript,display
WORK=Path('/kaggle/working')/f'giada_task17d_{uuid.uuid4().hex[:8]}'
REPO=WORK/'giada';TEACHER=WORK/'neuron_as_deep_net';WORK.mkdir(parents=True)
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER)],check=True)
subprocess.run(['git','-C',str(TEACHER),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert (REPO/'src/giada_teacher/roadmap_task17d_native_solver_mechanism.py').is_file()
print({'giada_revision':REVISION,'teacher_revision':'074c4666300a8ad246601dab179a97a6942f0f29'})

## Preflight NEURON
Codice, notebook e protocollo devono essere dello stesso commit. Non occorre aggiungere Dataset Kaggle: la Task 17d usa il teacher canonico.

In [ ]:
try:
 import neuron
except ModuleNotFoundError:
 subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7'],check=True)
 import neuron
assert neuron.__version__.split('+',1)[0]=='8.2.7',f'NEURON {neuron.__version__} non corrisponde a 8.2.7'
nrnivmodl=shutil.which('nrnivmodl') or str(Path(sys.executable).parent/'nrnivmodl')
assert Path(nrnivmodl).is_file() and shutil.which('gcc'),'Compilatore NEURON/gcc assente'
simulation_dir=TEACHER/'L5PC_NEURON_simulation'
if not list(simulation_dir.rglob('libnrnmech.so')):subprocess.run([nrnivmodl,'mods'],cwd=simulation_dir,check=True)
assert list(simulation_dir.rglob('libnrnmech.so')),'Meccanismi canonici non compilati'
print({'neuron':neuron.__version__,'canonical_mechanisms':'ready'})

## Matrice nativa appaiata
Cinque politiche CVode su tre condizioni note, più il controllo della cadenza del driver su due casi. La finestra è 60 ms. Il report è diagnostico anche se tutte le righe sono valide.

In [ ]:
OUTPUT=Path('/kaggle/working/artifacts')/f'giada_task17d_native_solver_{REVISION[:7]}_{uuid.uuid4().hex[:8]}'
command=[sys.executable,'-u',str(REPO/'scripts/run_roadmap_task17d.py'),'--repo',str(REPO),'--teacher',str(TEACHER),'--output',str(OUTPUT)]
completed=subprocess.run(command,check=False)
path=OUTPUT/('failure_report.json' if (OUTPUT/'failure_report.json').is_file() else 'final_report.json')
report=json.loads(path.read_text()) if path.is_file() else {'valid':False,'decision':'SUPERVISOR_INTERRUPTED','error':f'exit={completed.returncode}; controllare process.log'}
display({key:report.get(key) for key in ('valid','decision','gate_c_authorized','calcium_state_basename','policy_audit','paired_comparison_count','all_release_hashes_match_ultra','error_type','error')})
print({'output':str(OUTPUT),'returncode':completed.returncode,'next':'Scarica lo ZIP anche se il processo fallisce'})

## ZIP diagnostico
Scarica metriche, tracciati, hash dei rilasci e log. Gli array non vengono stampati nella pagina Kaggle.

In [ ]:
EXPORT=Path(tempfile.mkdtemp(prefix='task17d_download_',dir=WORK))
for name in ('final_report.json','failure_report.json','process_status.json','process.log','last_phase.json','paired_metrics.json','native_traces.json','release_hashes.json','trial_runtime_seconds.json'):
 source=OUTPUT/name
 if source.is_file():shutil.copy2(source,EXPORT/name)
archive=Path(shutil.make_archive(str(Path('/kaggle/working')/OUTPUT.name),'zip',EXPORT.parent,EXPORT.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})